### Overview

Porovnanie výsledkov ukazuje, že pre metódu SVM je optimálna hodnota σ = 1000. Ďalšie zvyšovanie hodnoty σ vedie k zníženiu kvality modelu

Pre prahové hodnoty 500 a 1000
Zo všetkých variantov sa ako najlepší ukázala základná model SVM bez škálovania
Najhoršie výsledky dosiahla model s vyvažovaním tried, ktorý napriek veľmi vysokému recallu mal extrémne nízku presnosť, čo podstatne znížilo celkovú efektívnosť klasifikácie.

Pre prah 1500
Zo všetkých variantov sa ako najlepší ukázal základný model SVM bez škálovania,
Naopak najhoršie výsledky  opäť dosiahol model s vyvažovaním tried

In [11]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.model_selection import RandomizedSearchCV



from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)

In [12]:
TW_500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=500/Twitter-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TW_1000= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1000/Twitter-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TW_1500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1500/Twitter-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_500.columns = columns
TW_1000.columns = columns
TW_1500.columns = columns


### 500

In [3]:
X = TW_500.drop("label", axis=1)
y = TW_500["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### 1.Baseline SVM without scaling

In [8]:
X = TW_500.drop("label", axis=1)
y = TW_500["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

svm_baseline = SVC(probability=True,random_state=42).fit(X_train, y_train)

y_pred_svm = svm_baseline.predict(X_test)
y_prob_svm = svm_baseline.predict_proba(X_test)[:, 1]

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
roc_auc_svm = roc_auc_score(y_test, y_prob_svm)

cm_svm = confusion_matrix(y_test, y_pred_svm)


print('1.Baseline SVM without scaling')
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)
print("ROC-AUC:", roc_auc_svm)

print("\nConfusion Matrix:")
print(cm_svm)

1.Baseline SVM without scaling
Accuracy: 0.9808826664771516
Precision: 0.7835365853658537
Recall: 0.35497237569060774
F1-score: 0.48859315589353614
ROC-AUC: 0.7491056959798559

Confusion Matrix:
[[27347    71]
 [  467   257]]


### 2.Baseline SVM with StandardScaler

In [7]:
pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1, max_iter=2000))
])

pipe_svm.fit(X_train, y_train)

y_pred = pipe_svm.predict(X_test)

print("Baseline SVM with StandardScaler")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Baseline SVM with StandardScaler
Accuracy: 0.9807760642456116
Precision: 0.8144329896907216
Recall: 0.32734806629834257
F1: 0.4669950738916256


### 3.Balanced SVM with scaling

In [9]:
pipe_svm_bal = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1, class_weight="balanced", max_iter=2000))
])

pipe_svm_bal.fit(X_train, y_train)

y_pred = pipe_svm_bal.predict(X_test)

print("Balanced SVM with scaling")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Balanced SVM with scaling
Accuracy: 0.927261744012508
Precision: 0.24068992551940416
Recall: 0.8480662983425414
F1: 0.3749618320610687


### 4.SVM with Grid Search

In [13]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(max_iter=2000))
])

param_dist = {"svm__C": [0.1, 1, 10]}

search = RandomizedSearchCV(pipeline,param_distributions=param_dist,n_iter=3,cv=3,scoring="f1",n_jobs=-1,random_state=42)

search.fit(X_train, y_train)

best_svm = search.best_estimator_
y_pred = best_svm.predict(X_test)

print("SVM with Grid Search")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

SVM with Grid Search
Accuracy: 0.9808115983227915
Precision: 0.804635761589404
Recall: 0.3356353591160221
F1: 0.47368421052631576


In [24]:
import pandas as pd

svm_500 = [
    {"Model": "SVM baseline", "Accuracy": 0.98088, "Precision": 0.7835, "Recall": 0.3550, "F1": 0.4886, "ROC-AUC": 0.7491},
    {"Model": "SVM + scaling", "Accuracy": 0.98078, "Precision": 0.8144, "Recall": 0.3273, "F1": 0.4670, "ROC-AUC": None},
    {"Model": "SVM balanced", "Accuracy": 0.92726, "Precision": 0.2407, "Recall": 0.8481, "F1": 0.3750, "ROC-AUC": None},
    {"Model": "SVM tuned", "Accuracy": 0.98081, "Precision": 0.8046, "Recall": 0.3356, "F1": 0.4737, "ROC-AUC": None}
]

df_svm_500 = pd.DataFrame(svm_500)
df_svm_500.sort_values(by="F1", ascending=False).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,SVM baseline,0.9809,0.7835,0.3550,0.4886,0.7491
3,SVM tuned,0.9808,0.8046,0.3356,0.4737,NaN
1,SVM + scaling,0.9808,0.8144,0.3273,0.4670,NaN
2,SVM balanced,0.9273,0.2407,0.8481,0.3750,NaN


### 1000

In [14]:
X = TW_1000.drop("label", axis=1)
y = TW_1000["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### 1.Baseline SVM without scaling

In [15]:
svm_baseline = SVC(probability=True,random_state=42).fit(X_train, y_train)

y_pred_svm = svm_baseline.predict(X_test)
y_prob_svm = svm_baseline.predict_proba(X_test)[:, 1]

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
roc_auc_svm = roc_auc_score(y_test, y_prob_svm)

cm_svm = confusion_matrix(y_test, y_pred_svm)


print('1.Baseline SVM without scaling')
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)
print("ROC-AUC:", roc_auc_svm)

print("\nConfusion Matrix:")
print(cm_svm)

1.Baseline SVM without scaling
Accuracy: 0.9945632861914576
Precision: 0.7733333333333333
Recall: 0.49361702127659574
F1-score: 0.6025974025974026
ROC-AUC: 0.8308015452540315

Confusion Matrix:
[[27873    34]
 [  119   116]]


### 2.Baseline SVM with StandardScaler

In [16]:
pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1, max_iter=2000))
])

pipe_svm.fit(X_train, y_train)

y_pred = pipe_svm.predict(X_test)

print("Baseline SVM with StandardScaler")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Baseline SVM with StandardScaler
Accuracy: 0.9939947409565774
Precision: 0.8055555555555556
Recall: 0.3702127659574468
F1: 0.5072886297376094


### 3.Balanced SVM with scaling

In [17]:
pipe_svm_bal = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1, class_weight="balanced", max_iter=2000))
])

pipe_svm_bal.fit(X_train, y_train)

y_pred = pipe_svm_bal.predict(X_test)

print("Balanced SVM with scaling")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Balanced SVM with scaling
Accuracy: 0.9627958211925236
Precision: 0.16884176182707994
Recall: 0.8808510638297873
F1: 0.28336755646817247


### 4.SVM with Grid Search

In [18]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(max_iter=2000))
])

param_dist = {"svm__C": [0.1, 1, 10]}

search = RandomizedSearchCV(pipeline,param_distributions=param_dist,n_iter=3,cv=3,scoring="f1",n_jobs=-1,random_state=42)

search.fit(X_train, y_train)

best_svm = search.best_estimator_
y_pred = best_svm.predict(X_test)

print("SVM with Grid Search")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

SVM with Grid Search
Accuracy: 0.9939947409565774
Precision: 0.8055555555555556
Recall: 0.3702127659574468
F1: 0.5072886297376094


In [26]:
import pandas as pd

svm_1000 = [
    {"Model": "SVM baseline", "Accuracy": 0.99456, "Precision": 0.7733, "Recall": 0.4936, "F1": 0.6026, "ROC-AUC": 0.8308},
    {"Model": "SVM + scaling", "Accuracy": 0.99399, "Precision": 0.8056, "Recall": 0.3702, "F1": 0.5073, "ROC-AUC": None},
    {"Model": "SVM balanced", "Accuracy": 0.96280, "Precision": 0.1688, "Recall": 0.8809, "F1": 0.2834, "ROC-AUC": None},
    {"Model": "SVM tuned", "Accuracy": 0.99399, "Precision": 0.8056, "Recall": 0.3702, "F1": 0.5073, "ROC-AUC": None}
]

df_svm_1000 = pd.DataFrame(svm_1000)
df_svm_1000

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,SVM baseline,0.99456,0.7733,0.4936,0.6026,0.8308
1,SVM + scaling,0.99399,0.8056,0.3702,0.5073,NaN
2,SVM balanced,0.96280,0.1688,0.8809,0.2834,NaN
3,SVM tuned,0.99399,0.8056,0.3702,0.5073,NaN


### 1500

In [19]:
X = TW_1500.drop("label", axis=1)
y = TW_1500["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### 1.Baseline SVM without scaling

In [20]:
svm_baseline = SVC(probability=True,random_state=42).fit(X_train, y_train)

y_pred_svm = svm_baseline.predict(X_test)
y_prob_svm = svm_baseline.predict_proba(X_test)[:, 1]

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
roc_auc_svm = roc_auc_score(y_test, y_prob_svm)

cm_svm = confusion_matrix(y_test, y_pred_svm)


print('1.Baseline SVM without scaling')
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)
print("ROC-AUC:", roc_auc_svm)

print("\nConfusion Matrix:")
print(cm_svm)

1.Baseline SVM without scaling
Accuracy: 0.9971217397484188
Precision: 0.7428571428571429
Recall: 0.2653061224489796
F1-score: 0.39097744360902253
ROC-AUC: 0.8775866786594827

Confusion Matrix:
[[28035     9]
 [   72    26]]


### 2.Baseline SVM with StandardScaler

In [21]:
pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1, max_iter=2000))
])

pipe_svm.fit(X_train, y_train)

y_pred = pipe_svm.predict(X_test)

print("Baseline SVM with StandardScaler")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Baseline SVM with StandardScaler
Accuracy: 0.9975126145973989
Precision: 0.7692307692307693
Recall: 0.40816326530612246
F1: 0.5333333333333333


### 3.Balanced SVM with scaling

In [22]:
pipe_svm_bal = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1, class_weight="balanced", max_iter=2000))
])

pipe_svm_bal.fit(X_train, y_train)

y_pred = pipe_svm_bal.predict(X_test)

print("Balanced SVM with scaling")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Balanced SVM with scaling
Accuracy: 0.9780754743799304
Precision: 0.13086770981507823
Recall: 0.9387755102040817
F1: 0.22971285892634208


### 4.SVM with Grid Search

In [23]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(max_iter=2000))
])

param_dist = {"svm__C": [0.1, 1, 10]}

search = RandomizedSearchCV(pipeline,param_distributions=param_dist,n_iter=3,cv=3,scoring="f1",n_jobs=-1,random_state=42)

search.fit(X_train, y_train)

best_svm = search.best_estimator_
y_pred = best_svm.predict(X_test)

print("SVM with Grid Search")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

SVM with Grid Search
Accuracy: 0.9975126145973989
Precision: 0.7692307692307693
Recall: 0.40816326530612246
F1: 0.5333333333333333


In [27]:
import pandas as pd

svm_1500 = [
    {"Model": "SVM baseline", "Accuracy": 0.99712, "Precision": 0.7429, "Recall": 0.2653, "F1": 0.3910, "ROC-AUC": 0.8776},
    {"Model": "SVM + scaling", "Accuracy": 0.99751, "Precision": 0.7692, "Recall": 0.4082, "F1": 0.5333, "ROC-AUC": None},
    {"Model": "SVM balanced", "Accuracy": 0.97808, "Precision": 0.1309, "Recall": 0.9388, "F1": 0.2297, "ROC-AUC": None},
    {"Model": "SVM tuned", "Accuracy": 0.99751, "Precision": 0.7692, "Recall": 0.4082, "F1": 0.5333, "ROC-AUC": None}
]

df_svm_1500 = pd.DataFrame(svm_1500)
df_svm_1500.sort_values(by="F1", ascending=False).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
1,SVM + scaling,0.9975,0.7692,0.4082,0.5333,NaN
3,SVM tuned,0.9975,0.7692,0.4082,0.5333,NaN
0,SVM baseline,0.9971,0.7429,0.2653,0.3910,0.8776
2,SVM balanced,0.9781,0.1309,0.9388,0.2297,NaN
